In [1]:
# pip install requests pandas python-dateutil
import requests, math, time
import pandas as pd
from datetime import datetime

FMP_BASE = "https://financialmodelingprep.com/api/v3"

def _get_json(url, params=None, retries=2, sleep=0.7):
    for i in range(retries+1):
        try:
            r = requests.get(url, params=params, timeout=20)
            if r.status_code == 200:
                return r.json()
        except Exception:
            pass
        time.sleep(sleep)
    return None

def get_income_q(ticker, api_key, limit=1):
    """최근 분기 손익계산서(Quarterly): revenue, operatingIncome"""
    url = f"{FMP_BASE}/income-statement/{ticker}"
    js = _get_json(url, {"period":"quarter", "limit":limit, "apikey":api_key})
    if not js:
        return None
    # 첫 행만 사용(가장 최근 분기)
    row = js[0]
    return {
        "revenue": row.get("revenue"),
        "operatingIncome": row.get("operatingIncome"),
        "date": row.get("date")
    }

def get_roa_ttm(ticker, api_key):
    """ROA 우선 TTM에서, 없으면 최신 key-metrics에서 보완"""
    # 1) TTM
    url = f"{FMP_BASE}/key-metrics-ttm/{ticker}"
    js = _get_json(url, {"apikey": api_key})
    if js and isinstance(js, list) and len(js) > 0:
        roa = js[0].get("roaTTM")
        if roa is not None:
            return roa
    # 2) 일반 key-metrics (분기/연간 혼재 → 가장 최신 값 사용)
    url = f"{FMP_BASE}/key-metrics/{ticker}"
    js = _get_json(url, {"period":"quarter", "limit":1, "apikey": api_key})
    if js and isinstance(js, list) and len(js) > 0:
        roa = js[0].get("roa")
        if roa is not None:
            return roa
    return None

def get_market_cap(ticker, api_key):
    """시가총액: quote에서 marketCap 사용"""
    url = f"{FMP_BASE}/quote/{ticker}"
    js = _get_json(url, {"apikey": api_key})
    if js and isinstance(js, list) and len(js) > 0:
        return js[0].get("marketCap")
    return None

def format_billions(x):
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return None
    try:
        return round(x / 1e9, 2)
    except Exception:
        return x

def collect_snapshot(tickers, api_key):
    records = []
    for tk in tickers:
        inc = get_income_q(tk, api_key, limit=1)
        roa = get_roa_ttm(tk, api_key)
        mcap = get_market_cap(tk, api_key)

        if inc:
            rev = inc["revenue"]
            oi  = inc["operatingIncome"]
            opm = (oi / rev * 100.0) if (oi is not None and rev and rev != 0) else None
            date = inc["date"]
        else:
            rev = oi = opm = date = None

        records.append({
            "Ticker": tk,
            "Quarter(Date)": date,
            "Revenue (USD Bn)": format_billions(rev),
            "Operating Income (USD Bn)": format_billions(oi),
            "Operating Margin (%)": None if opm is None else round(opm, 1),
            "ROA (TTM, %)": None if roa is None else round(roa*100 if roa < 1 else roa, 2),
            "Market Cap (USD Bn)": format_billions(mcap)
        })
    df = pd.DataFrame(records)
    # 보기 좋게 정렬
    cols = ["Ticker","Quarter(Date)","Revenue (USD Bn)","Operating Income (USD Bn)",
            "Operating Margin (%)","ROA (TTM, %)","Market Cap (USD Bn)"]
    return df[cols]

# ===== 사용 예시 =====
# API 키 입력
API_KEY = "hT0gAk87j9xZx4PlBApvBqfVL5IahvgV"

# 포트폴리오 예시 (질문에 맞춤)
tickers = ["LLY"]

df = collect_snapshot(tickers, API_KEY)
print(df.to_string(index=False))

# 필요 시 CSV로 저장
# df.to_csv("fmp_quarter_snapshot.csv", index=False, encoding="utf-8")


Ticker Quarter(Date)  Revenue (USD Bn)  Operating Income (USD Bn)  Operating Margin (%) ROA (TTM, %)  Market Cap (USD Bn)
   LLY    2026-06-30             22.97                       8.98                  39.1         None              1206.13


#### 시계열 데이터 추출

In [2]:
"""
FMP 재무 DB 유틸리티 v5
========================
실제 DB 테이블 구조 (Long format)
  컬럼: date | report_date | ticker | period | date_month | item | value | fs_name

  ※ item 컬럼에 compustat 단축명이 그대로 저장됨
     IS: sale, cogs, gp, xrd, xsga, idit, xint, dp, ebitda, opiti, ni, eps ...
     BS: at, seq, debt, cash, rect, invt, lt, dltt ...
     CF: oancf, capx, fcf, dv, sbc ...

  → get_timeseries("MU", ["sale","ni","ebitda"], "US_IS_from_FMP") 처럼
    item명을 그대로 입력하면 됩니다. alias 변환 불필요.

테이블:
  US_IS_from_FMP  손익계산서
  US_BS_from_FMP  재무상태표
  US_CF_from_FMP  현금흐름표

의존: sqlalchemy, pymysql, pandas
"""

import pandas as pd
from sqlalchemy import create_engine, text, inspect as sa_inspect
from DATA.config import get_db_host


# ============================================================
# 0) DB 연결
# ============================================================

DB_INFO = {
    "user":     "stox7412",
    "password": "Apt106503!~",
    "host":     get_db_host(),
    "port":     "3307",
    "database": "investar",
}

TABLE_IS = "US_IS_from_FMP"
TABLE_BS = "US_BS_from_FMP"
TABLE_CF = "US_CF_from_FMP"


def make_engine(db_info: dict = None):
    info = db_info or DB_INFO
    conn_str = (
        f"mysql+pymysql://{info['user']}:{info['password']}"
        f"@{info['host']}:{info['port']}/{info['database']}"
    )
    return create_engine(conn_str, pool_pre_ping=True)


# ============================================================
# 1) 항목명 설명 사전 (item값 = compustat 단축명 그대로)
# ============================================================

ITEM_DESC = {
    # IS (손익계산서)
    "sale":      "매출액",
    "cogs":      "매출원가",
    "gp":        "매출총이익",
    "xrd":       "연구개발비",
    "xsga":      "판매관리비(SGA)",
    "idit":      "감가상각포함이자비용",
    "xint":      "이자비용",
    "dp":        "감가상각비",
    "ebitda":    "EBITDA",
    "xopr":      "영업비용합계",
    "opiti":     "영업이익",
    "opir":      "영업이익률",
    "pi":        "세전이익",
    "pir":       "세전이익률",
    "txt":       "법인세",
    "ni":        "순이익",
    "nir":       "순이익률",
    "eps":       "EPS(기본)",
    "epsdi":     "EPS(희석)",
    "shrout":    "발행주식수(기본)",
    "shroutdi":  "발행주식수(희석)",
    # BS (재무상태표)
    "cash":      "현금및현금성자산",
    "rect":      "매출채권",
    "invt":      "재고자산",
    "act":       "유동자산합계",
    "ppent":     "유형자산(순)",
    "gdwl":      "영업권",
    "at":        "총자산",
    "ap":        "매입채무",
    "dlc":       "단기차입금",
    "lct":       "유동부채합계",
    "dltt":      "장기차입금",
    "lt":        "총부채",
    "seq":       "자기자본",
    "teq":       "총자본",
    "debt":      "총차입금",
    "netdebt":   "순차입금",
    "re":        "이익잉여금",
    # CF (현금흐름표)
    "oancf":     "영업현금흐름",
    "capx":      "설비투자(CAPEX)",
    "fcf":       "잉여현금흐름(FCF)",
    "ivncf":     "투자현금흐름",
    "fincf":     "재무현금흐름",
    "dv":        "배당금지급",
    "sbc":       "주식보상비용",
    "prstkc":    "자사주매입",
}

# 테이블별 주요 항목 목록 (참고용)
ITEMS_IS = ["sale","cogs","gp","xrd","xsga","idit","xint","dp","ebitda",
            "xopr","opiti","opir","pi","pir","txt","ni","nir",
            "eps","epsdi","shrout","shroutdi"]

ITEMS_BS = ["cash","rect","invt","act","ppent","gdwl","at",
            "ap","dlc","lct","dltt","lt","seq","teq","debt","netdebt","re"]

ITEMS_CF = ["oancf","capx","fcf","ivncf","fincf","dv","sbc","prstkc"]


# ============================================================
# 2) DB 구조 확인 함수
# ============================================================

def show_items(table_name: str, engine=None, limit: int = 200) -> pd.DataFrame:
    """
    테이블의 item 컬럼에 실제 저장된 항목 목록과 설명을 출력합니다.
    이 목록의 값을 get_timeseries()의 columns 인수에 그대로 사용하세요.

    사용 예:
        show_items("US_IS_from_FMP")
        show_items("US_BS_from_FMP")
        show_items("US_CF_from_FMP")
    """
    eng = engine or make_engine()
    with eng.connect() as conn:
        rows = conn.execute(
            text(f"SELECT DISTINCT item FROM `{table_name}` LIMIT :lim"),
            {"lim": limit}
        ).fetchall()

    items = sorted([r[0] for r in rows])
    result_rows = []
    for item in items:
        desc = ITEM_DESC.get(item, "")
        result_rows.append({"item(입력값)": item, "설명": desc})

    df = pd.DataFrame(result_rows)
    print(f"\n[{table_name}]  저장된 item 종류: {len(df)}개")
    print("─" * 45)
    print(df.to_string(index=False))
    return df


def show_item_desc(table: str = None) -> pd.DataFrame:
    """
    테이블별 항목명과 한글 설명 목록을 출력합니다.

    Parameters
    ----------
    table : "IS" / "BS" / "CF" / None(전체)

    사용 예:
        show_item_desc()        # 전체
        show_item_desc("IS")    # 손익계산서만
        show_item_desc("BS")    # 재무상태표만
        show_item_desc("CF")    # 현금흐름표만
    """
    mapping = {
        "IS": (ITEMS_IS, TABLE_IS),
        "BS": (ITEMS_BS, TABLE_BS),
        "CF": (ITEMS_CF, TABLE_CF),
    }

    if table:
        targets = {table.upper(): mapping[table.upper()]}
    else:
        targets = mapping

    rows = []
    for tbl_key, (item_list, tbl_name) in targets.items():
        for item in item_list:
            rows.append({
                "item(입력값)": item,
                "설명":         ITEM_DESC.get(item, ""),
                "테이블":       tbl_name,
            })

    df = pd.DataFrame(rows)
    header = f"항목 목록  ({table.upper() if table else '전체'})"
    print(f"\n{'='*60}\n {header}\n{'='*60}")
    print(df.to_string(index=False))
    return df


def inspect_db_columns(table_name: str = None, engine=None) -> dict:
    """
    테이블의 물리적 컬럼 구조를 출력합니다.

    사용 예:
        inspect_db_columns("US_IS_from_FMP")
    """
    eng = engine or make_engine()
    insp = sa_inspect(eng)
    target_tables = [table_name] if table_name else insp.get_table_names()

    result = {}
    for tbl in target_tables:
        try:
            cols = [c["name"] for c in insp.get_columns(tbl)]
            result[tbl] = cols
            print(f"\n[{tbl}]  물리 컬럼: {cols}")
        except Exception as e:
            result[tbl] = [f"ERROR: {e}"]
            print(f"\n[{tbl}]  ERROR: {e}")
    return result


# ============================================================
# 3) 단일 ticker 시계열 추출 (핵심 함수)
# ============================================================

def get_timeseries(
    ticker:      str,
    columns,                          # item명 str 또는 list (sale, ni, at 등)
    table_name:  str,
    engine=None,
    date_col:    str  = "date",
    ticker_col:  str  = "ticker",
    item_col:    str  = "item",
    value_col:   str  = "value",
    start_date:  str  = None,         # "YYYY-MM-DD"
    end_date:    str  = None,
    use_report_date: bool = False,    # True → report_date 기준으로 날짜 필터
    as_billions: bool = False,        # True → 10억 달러(B) 단위 변환
) -> pd.DataFrame:
    """
    특정 ticker의 재무항목 시계열을 DB에서 추출합니다.
    Long format → Wide format(index=date, columns=item명) 으로 반환합니다.

    Parameters
    ----------
    ticker      : 종목 코드 (예: "MU", "AAPL", "GOOG")
    columns     : item명 str 또는 list. show_items()에서 확인한 값 그대로 사용.
                  IS 예: ["sale","ni","ebitda","opiti","gp","eps"]
                  BS 예: ["at","seq","debt","cash","rect","invt"]
                  CF 예: ["oancf","capx","fcf","dv"]
    table_name  : "US_IS_from_FMP" / "US_BS_from_FMP" / "US_CF_from_FMP"
    date_col    : 날짜 컬럼명 (기본 "date")
    ticker_col  : 티커 컬럼명 (기본 "ticker")
    item_col    : 항목명 컬럼 (기본 "item")
    value_col   : 값 컬럼 (기본 "value")
    start_date  : "YYYY-MM-DD". None이면 전체.
    end_date    : None이면 전체.
    use_report_date : True이면 report_date 기준으로 날짜 필터 적용
    as_billions : True이면 10억 달러(B) 단위로 변환

    Returns
    -------
    pd.DataFrame  (index=date, columns=item명)

    사용 예:
        # 손익계산서 IS
        df = get_timeseries("MU", ["sale","ni","ebitda"], "US_IS_from_FMP",
                            start_date="2018-01-01", as_billions=True)

        # 재무상태표 BS
        df = get_timeseries("MU", ["at","seq","debt","cash"], "US_BS_from_FMP",
                            as_billions=True)

        # 현금흐름표 CF
        df = get_timeseries("MU", ["oancf","capx","fcf"], "US_CF_from_FMP",
                            as_billions=True)

        # 단일 항목
        df = get_timeseries("AAPL", "sale", "US_IS_from_FMP", as_billions=True)
    """
    eng = engine or make_engine()

    if isinstance(columns, str):
        columns = [columns]

    # item IN (...) 필터
    item_ph = ", ".join([f":item{i}" for i in range(len(columns))])
    params  = {"ticker": ticker}
    params.update({f"item{i}": it for i, it in enumerate(columns)})

    filter_date = "report_date" if use_report_date else date_col
    where = [
        f"`{ticker_col}` = :ticker",
        f"`{item_col}` IN ({item_ph})",
    ]
    if start_date:
        where.append(f"`{filter_date}` >= :start_date")
        params["start_date"] = start_date
    if end_date:
        where.append(f"`{filter_date}` <= :end_date")
        params["end_date"] = end_date

    sql = text(
        f"SELECT `{date_col}`, `{item_col}`, `{value_col}` "
        f"FROM `{table_name}` "
        f"WHERE {' AND '.join(where)} "
        f"ORDER BY `{date_col}`"
    )

    with eng.connect() as conn:
        df_long = pd.read_sql(sql, conn, params=params)

    if df_long.empty:
        print(f"[경고] {ticker} — 데이터 없음 "
              f"(테이블: {table_name}, 항목: {columns})")
        return df_long

    df_long[date_col]  = pd.to_datetime(df_long[date_col])
    df_long[value_col] = pd.to_numeric(df_long[value_col], errors="coerce")

    # Long → Wide pivot
    df_wide = (
        df_long
        .pivot_table(index=date_col, columns=item_col,
                     values=value_col, aggfunc="last")
        .sort_index()
    )
    df_wide.columns.name = None

    # 요청했지만 결과에 없는 항목 경고
    missing = [c for c in columns if c not in df_wide.columns]
    if missing:
        print(f"[경고] 다음 항목이 {table_name}에 없습니다: {missing}")
        print(f"       show_items('{table_name}') 로 실제 항목 확인 후 재시도하세요.")

    # 요청한 순서대로 컬럼 정렬
    exist_cols = [c for c in columns if c in df_wide.columns]
    df_wide = df_wide[exist_cols]

    # 중복 날짜 제거
    df_wide = df_wide[~df_wide.index.duplicated(keep="last")]

    if as_billions:
        num_cols = df_wide.select_dtypes(include="number").columns
        df_wide[num_cols] = (df_wide[num_cols] / 1e9).round(3)

    unit_str = " (단위: B USD)" if as_billions else ""
    print(f"[{ticker}] {len(df_wide)}행{unit_str}  "
          f"({df_wide.index.min().date()} ~ {df_wide.index.max().date()})  "
          f"컬럼: {list(df_wide.columns)}")
    return df_wide


# ============================================================
# 4) 여러 ticker 동시 추출
# ============================================================

def get_multi_ticker_ts(
    tickers,
    columns,
    table_name:  str,
    engine=None,
    date_col:    str  = "date",
    ticker_col:  str  = "ticker",
    item_col:    str  = "item",
    value_col:   str  = "value",
    start_date:  str  = None,
    end_date:    str  = None,
    output:      str  = "wide",   # "wide" | "long"
    as_billions: bool = False,
) -> pd.DataFrame:
    """
    여러 ticker를 한 번에 조회합니다.

    Parameters
    ----------
    output : "wide" → index=date, columns=ticker (단일 항목 권장)
             "long" → ticker/date/item/value 형식 (복수 항목 가능)

    사용 예:
        # 매출 wide 비교 (컬럼=티커)
        wide = get_multi_ticker_ts(["AAPL","MU","NVDA"], "sale",
                                    "US_IS_from_FMP", as_billions=True)

        # 복수 항목 long 형식
        long = get_multi_ticker_ts(["AAPL","MU"], ["sale","ni","opiti"],
                                    "US_IS_from_FMP", output="long",
                                    as_billions=True)

        # BS 총자산 비교
        wide = get_multi_ticker_ts(["GOOG","TSLA","MSFT"], "at",
                                    "US_BS_from_FMP", as_billions=True)
    """
    eng = engine or make_engine()

    if isinstance(tickers, str):
        tickers = [tickers]
    if isinstance(columns, str):
        columns = [columns]

    ticker_ph = ", ".join([f":t{i}" for i in range(len(tickers))])
    item_ph   = ", ".join([f":item{i}" for i in range(len(columns))])

    params = {}
    params.update({f"t{i}":    tk for i, tk in enumerate(tickers)})
    params.update({f"item{i}": it for i, it in enumerate(columns)})

    where = [
        f"`{ticker_col}` IN ({ticker_ph})",
        f"`{item_col}` IN ({item_ph})",
    ]
    if start_date:
        where.append(f"`{date_col}` >= :start_date")
        params["start_date"] = start_date
    if end_date:
        where.append(f"`{date_col}` <= :end_date")
        params["end_date"] = end_date

    sql = text(
        f"SELECT `{date_col}`, `{ticker_col}`, `{item_col}`, `{value_col}` "
        f"FROM `{table_name}` "
        f"WHERE {' AND '.join(where)} "
        f"ORDER BY `{ticker_col}`, `{date_col}`"
    )

    with eng.connect() as conn:
        df_long = pd.read_sql(sql, conn, params=params)

    if df_long.empty:
        print("[경고] 데이터 없음")
        return df_long

    df_long[date_col]  = pd.to_datetime(df_long[date_col])
    df_long[value_col] = pd.to_numeric(df_long[value_col], errors="coerce")

    if as_billions:
        df_long[value_col] = (df_long[value_col] / 1e9).round(3)

    unit_str = " (B USD)" if as_billions else ""

    # wide: 단일 항목, 컬럼=티커
    if output == "wide" and len(columns) == 1:
        item = columns[0]
        df_filt = df_long[df_long[item_col] == item]
        wide = (
            df_filt
            .pivot_table(index=date_col, columns=ticker_col,
                         values=value_col, aggfunc="last")
            .sort_index()
        )
        wide.columns.name = None
        print(f"[wide] '{item}'{unit_str}  "
              f"{len(wide)}행 x {len(wide.columns)}종목  "
              f"({wide.index.min().date()} ~ {wide.index.max().date()})")
        return wide

    # long 그대로 반환
    print(f"[long]{unit_str}  {len(df_long)}행  "
          f"티커 {df_long[ticker_col].nunique()}개  "
          f"항목 {df_long[item_col].nunique()}개")
    return df_long


# ============================================================
# 5) 예측 결과 테이블 조회
# ============================================================

def get_forecast_ts(
    ticker:      str,
    target_col:  str  = "sale",
    table_name:  str  = "us_fs_forecast_data",
    engine=None,
    latest_only: bool = True,
) -> pd.DataFrame:
    """
    SARIMA 예측 결과에서 특정 ticker의 예측 시계열을 가져옵니다.

    사용 예:
        fc = get_forecast_ts("MU",   target_col="sale")
        fc = get_forecast_ts("AAPL", target_col="opiti")
    """
    eng = engine or make_engine()

    with eng.connect() as conn:
        stored = [r[0] for r in conn.execute(
            text(f"SELECT DISTINCT target_col FROM `{table_name}` LIMIT 30")
        ).fetchall()]

        if target_col not in stored:
            print(f"[경고] '{target_col}' 없음. 저장된 항목: {stored}")
            return pd.DataFrame()

        if latest_only:
            latest_dt = conn.execute(
                text(f"SELECT MAX(prediction_date) FROM `{table_name}` "
                     f"WHERE ticker=:t AND target_col=:tc"),
                {"t": ticker, "tc": target_col}
            ).scalar()
            if latest_dt is None:
                print(f"[경고] {ticker}/{target_col} 예측 없음")
                return pd.DataFrame()
            df = pd.read_sql(
                text(f"SELECT forecast_date, forecast_value, lower_ci, upper_ci, "
                     f"model_params, transform_type, prediction_date "
                     f"FROM `{table_name}` "
                     f"WHERE ticker=:t AND target_col=:tc AND prediction_date=:pd "
                     f"ORDER BY forecast_date"),
                conn, params={"t": ticker, "tc": target_col, "pd": latest_dt}
            )
        else:
            df = pd.read_sql(
                text(f"SELECT forecast_date, forecast_value, lower_ci, upper_ci, "
                     f"model_params, transform_type, prediction_date "
                     f"FROM `{table_name}` "
                     f"WHERE ticker=:t AND target_col=:tc "
                     f"ORDER BY prediction_date DESC, forecast_date"),
                conn, params={"t": ticker, "tc": target_col}
            )

    if df.empty:
        return df

    df["forecast_date"] = pd.to_datetime(df["forecast_date"])
    df = df.set_index("forecast_date").sort_index()
    print(f"[예측] {ticker}/{target_col}  {len(df)}분기  "
          f"({df.index.min().date()} ~ {df.index.max().date()})")
    return df


# ============================================================
# 6) 사용 가이드
# ============================================================

def help_guide():
    print("""
================================================================
FMP DB 시계열 유틸리티 v5  —  사용 가이드
================================================================
실제 DB 구조 (Long format)
  date | report_date | ticker | period | date_month | item | value | fs_name
  item 컬럼 = compustat 단축명 (sale, cogs, gp, at, oancf ...)
  value 컬럼 = 숫자값 (원 단위, as_billions=True 로 B 단위 변환 가능)

테이블
  US_IS_from_FMP  손익계산서  (fs_name='is')
  US_BS_from_FMP  재무상태표  (fs_name='bs')
  US_CF_from_FMP  현금흐름표  (fs_name='cf')

─────────────────────────────────────────────────────────────
[Step 0]  사용 가능한 item 확인
    show_items("US_IS_from_FMP")   # 실제 DB 저장 항목 목록
    show_items("US_BS_from_FMP")
    show_items("US_CF_from_FMP")
    show_item_desc("IS")           # 설명 포함 목록
    show_item_desc("BS")
    show_item_desc("CF")

[Step 1]  단일 ticker 시계열
    # 손익계산서
    df = get_timeseries("MU", ["sale","ni","ebitda"], "US_IS_from_FMP",
                        start_date="2018-01-01", as_billions=True)
    # 재무상태표
    df = get_timeseries("MU", ["at","seq","debt","cash"], "US_BS_from_FMP",
                        as_billions=True)
    # 현금흐름표
    df = get_timeseries("MU", ["oancf","capx","fcf"], "US_CF_from_FMP",
                        as_billions=True)

[Step 2]  여러 ticker — wide (단일 항목, 컬럼=티커)
    wide = get_multi_ticker_ts(["AAPL","MU","NVDA"], "sale",
                                "US_IS_from_FMP", as_billions=True)

[Step 3]  여러 ticker — long (복수 항목)
    long = get_multi_ticker_ts(["AAPL","MU"], ["sale","ni","opiti"],
                                "US_IS_from_FMP", output="long",
                                as_billions=True)

[Step 4]  SARIMA 예측 결과 조회
    fc = get_forecast_ts("MU",   target_col="sale")
    fc = get_forecast_ts("AAPL", target_col="opiti")

─────────────────────────────────────────────────────────────
주요 item명 (테이블 구분 중요)
  IS: sale  cogs  gp  xrd  xsga  idit  xint  dp
      ebitda  opiti  ni  eps  epsdi
  BS: cash  rect  invt  act  ppent  at
      ap  dlc  lct  dltt  lt  seq  debt  netdebt  re
  CF: oancf  capx  fcf  ivncf  fincf  dv  sbc  prstkc
================================================================
""")

In [8]:
df = get_timeseries("PM", ["sale", "ni"], "US_IS_from_FMP", start_date="2026-01-01", as_billions=True)

[경고] PM — 데이터 없음 (테이블: US_IS_from_FMP, 항목: ['sale', 'ni'])


In [5]:
df

,date,item,value


In [5]:
# sample_data_save_path = r'C:\Users\82108\OneDrive\INVESTMENT\미국주식\raw_data\mu_sales.csv'
#
# df.to_csv(sample_data_save_path)